**Importing The Dependencies**

In [274]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBRFClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

**Data Collection & Loading**

In [275]:
# load the clean dataset into a pandas DataFrame
df = pd.read_csv("../dataset/clean dataset/final_telco_customer_churn_cleaned_dataset.csv")

In [276]:
df.head()

,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,...,contract,paper_less_billing,payment_method,monthly_charges,total_charges,churn,customer_profile,churn_numeric,tenure_groups,total_services
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,Partner Only,0,0-1 Year,2
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,No,Single / No Family,0,2-4 Years,4
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,Single / No Family,1,0-1 Year,4
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,Single / No Family,0,2-4 Years,4
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,Single / No Family,1,0-1 Year,2


In [277]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   gender              7043 non-null   object 
 1   senior_citizen      7043 non-null   int64  
 2   partner             7043 non-null   object 
 3   dependents          7043 non-null   object 
 4   tenure              7043 non-null   int64  
 5   phone_service       7043 non-null   object 
 6   multiple_lines      7043 non-null   object 
 7   internet_service    7043 non-null   object 
 8   online_security     7043 non-null   object 
 9   online_backup       7043 non-null   object 
 10  device_protection   7043 non-null   object 
 11  tech_support        7043 non-null   object 
 12  streaming_tv        7043 non-null   object 
 13  streaming_movies    7043 non-null   object 
 14  contract            7043 non-null   object 
 15  paper_less_billing  7043 non-null   object 
 16  paymen

In [278]:
# drop these columns since they may cause data leakage
df = df.drop(columns = ['churn_numeric', 'tenure_groups','customer_profile'])

In [279]:
df.head()

,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,...,tech_support,streaming_tv,streaming_movies,contract,paper_less_billing,payment_method,monthly_charges,total_charges,churn,total_services
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,2
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,4
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,4
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2


**Splitting Data into Features and Target**

In [280]:
# splitting the features and target
X = df.drop(columns = ["churn"])
Y = df["churn"]

**Splitting Data Into Training And Testing Data**

In [281]:
# split training and testing data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state = 2)

In [282]:
print(X.shape, X_train, X_test)

(7043, 20)       gender  senior_citizen partner dependents  tenure phone_service  \
4169  Female               0      No         No      15           Yes   
3571  Female               0      No         No      35           Yes   
1352    Male               0     Yes        Yes      72            No   
1278    Male               0     Yes        Yes      36            No   
938   Female               0     Yes         No      65           Yes   
...      ...             ...     ...        ...     ...           ...   
6443  Female               0      No         No       9           Yes   
3606  Female               0     Yes         No      70           Yes   
5704  Female               0     Yes         No      19           Yes   
6637    Male               0     Yes         No      69           Yes   
2575  Female               0      No         No      35           Yes   

        multiple_lines internet_service      online_security  \
4169                No      Fiber optic         

In [283]:
print(Y_train.shape)
print(Y_train.value_counts())

(5634,)
churn
No     4113
Yes    1521
Name: count, dtype: int64


**Data Preprocessing**

**Label Encoding Of Target Column**

In [284]:
# load the Label Encoder
label_encoder = LabelEncoder()

# encode target(y) using test using LabelEncoder
Y_train_encoded = label_encoder.fit_transform(Y_train)
Y_test_encoded = label_encoder.transform(Y_test)

print(Y_train_encoded)

[1 0 0 ... 0 0 0]


**One Hot Encoding Of Feature Column**

In [285]:
# Define categorical feature groups
cats_col = ['gender', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'streaming_tv', 'streaming_movies', 'contract', 'paper_less_billing', 'payment_method']

ohe_encoder = OneHotEncoder(drop='first', sparse_output=False)

X_train_encoded = ohe_encoder.fit_transform(X_train[cats_col])
X_test_encoded = ohe_encoder.transform(X_test[cats_col])

encoded_cols = ohe_encoder.get_feature_names_out(cats_col)

X_train_encoder_df = pd.DataFrame(X_train_encoded, columns=encoded_cols, index=X_train.index)
X_test_encoder_df = pd.DataFrame(X_test_encoded, columns=encoded_cols, index=X_test.index)

**Standard Scaler**

In [286]:
num_cols = ['tenure', 'senior_citizen', 'monthly_charges', 'total_charges']

scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train[num_cols])
X_test_scale = scaler.transform(X_test[num_cols])

X_train_scale_df = pd.DataFrame(X_train_scale, columns= num_cols, index=X_train.index)
X_test_scale_df = pd.DataFrame(X_test_scale, columns= num_cols, index=X_test.index)

X_train_final = pd.concat([X_train_scale_df, X_train_encoder_df], axis=1)
X_test_final = pd.concat([X_test_scale_df, X_test_encoder_df], axis=1)

print("Data is now encoded and scaled. Preview of first 5 rows")
X_train_final.head()

Data is now encoded and scaled. Preview of first 5 rows


,tenure,senior_citizen,monthly_charges,total_charges,gender_Male,partner_Yes,dependents_Yes,phone_service_Yes,multiple_lines_No phone service,multiple_lines_Yes,...,streaming_tv_No internet service,streaming_tv_Yes,streaming_movies_No internet service,streaming_movies_Yes,contract_One year,contract_Two year,paper_less_billing_Yes,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check
4169,-0.705310,-0.443207,1.206336,-0.363074,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3571,0.108051,-0.443207,-0.095544,-0.082258,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1352,1.612768,-0.443207,-0.133785,0.995974,1.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1278,0.148719,-0.443207,-0.142099,-0.021400,1.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
938,1.328092,-0.443207,1.432461,2.125590,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0


**Synthetic Minority Oversampling Technique(SMOTE)**

In [287]:
smote = SMOTE(random_state=2)

## **Model Selection & Development**

In [288]:
# list of models 

models = [LogisticRegression(random_state=2), RandomForestClassifier(random_state = 2), DecisionTreeClassifier(random_state = 2), XGBRFClassifier(random_state = 2)]

def compare_models_cross_validation():
    # Cross validating with default hyperparameters
    for model in models:

        imb_pipeline = make_pipeline(smote, model)

        cv_score = cross_val_score(imb_pipeline, X_train_final, Y_train_encoded, cv=5)
        f1_cv_score = cross_val_score(imb_pipeline, X_train_final, Y_train_encoded, cv=5, scoring='f1')
        mean_accuracy = round(sum(cv_score)/len(cv_score),2)
        f1_mean_accuracy = round(sum(f1_cv_score)/len(f1_cv_score),2)
        

        # CLean up the model name string for printing
        model_name = model.__class__.__name__

        print(f"Cross-Validation Accuracies for the {model_name} : {cv_score}")
        print(f"Accuracy Score of {mean_accuracy}%")
        print(f"Mean F1 Score of {f1_mean_accuracy}%")
        print("*" * 150)

compare_models_cross_validation()

Cross-Validation Accuracies for the LogisticRegression : [0.72493345 0.75598935 0.74889086 0.75598935 0.76198934]
Accuracy Score of 0.75%
Mean F1 Score of 0.63%
******************************************************************************************************************************************************
Cross-Validation Accuracies for the RandomForestClassifier : [0.76220053 0.7826087  0.78793256 0.77462289 0.79396092]
Accuracy Score of 0.78%
Mean F1 Score of 0.6%
******************************************************************************************************************************************************
Cross-Validation Accuracies for the DecisionTreeClassifier : [0.71517303 0.73558119 0.71251109 0.72493345 0.72024867]
Accuracy Score of 0.72%
Mean F1 Score of 0.52%
******************************************************************************************************************************************************
Cross-Validation Accuracies for the XGBRFClassifier : [0.

**For this dataset, XGBRF Classifier has the highest accuracy and F-1 score wtih default hyperparamters compared to other models.**

In [289]:
xgb_model = XGBRFClassifier(random_state=2, eval_metric = 'logloss')

In [290]:
tuning_pipeline = make_pipeline(smote, xgb_model)

**Applying GridSearchCV**

In [291]:
hyperparameter_grid = {
        'xgbrfclassifier__n_estimators' : [10, 50, 100],
        'xgbrfclassifier__max_depth' : [3, 5, 7],
        'xgbrfclassifier__learning_rate' : [0.05, 0.1, 0.2]
    }

grid_search = GridSearchCV(estimator = tuning_pipeline, param_grid = hyperparameter_grid, cv=5, scoring = 'f1')

print("Optimizing XGBoost Random Forest parameters...")
grid_search.fit(X_train_final, Y_train_encoded)

print('=' * 120)
print(f"Best Hyperparamters: {grid_search.best_params_}")
print(f"Optimized Cross-Validation F-1 Score: {grid_search.best_score_}")
print("=" * 120)

Optimizing XGBoost Random Forest parameters...
Best Hyperparamters: {'xgbrfclassifier__learning_rate': 0.05, 'xgbrfclassifier__max_depth': 7, 'xgbrfclassifier__n_estimators': 100}
Optimized Cross-Validation F-1 Score: 0.6323774100625139


Model Training

In [292]:
model = XGBRFClassifier(random_state = 2, learning_rate = 0.05, max_depth = 7, n_estimators = 100)

In [293]:
model.fit(X_train_final, Y_train_encoded)

,learning_rate,0.05
,subsample,0.8
,colsample_bynode,0.8
,reg_lambda,1e-05
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bytree,None
,device,None
